# So sánh TSR trên 3 instance, mỗi instance 50 tác vụ

Chọn runtime **CPU** và chạy ô bên dưới. Notebook tải mã benchmark và dữ liệu từ hai commit cố định trên GitHub.

| Instance | Tác vụ | Robot |
|---|---:|---:|
| LC101-n50 | 50 | 10 |
| LR101-n50 | 50 | 10 |
| LRC101-n50 | 50 | 10 |

Mỗi bộ dùng depot và các tác vụ ID **1–50** từ bộ CSV tương ứng của Thảo. `thao_original` và `thao_elite` dùng cùng nghiệm greedy ban đầu trên từng instance. Năm seed **0–4** chạy song song trong mỗi nhóm instance và thuật toán, với **600 giây CPU mỗi seed**. Tổng cộng có 30 lượt chạy.

Bật **SAVE_TO_DRIVE** để giữ các seed đã hoàn thành và chạy tiếp với cùng cấu hình. ZIP kết quả có `paired_results.csv` theo seed và `comparison_summary.csv` tổng hợp từng instance.


In [ ]:
# @title Chạy benchmark PDP-CR
TIME_LIMIT = 600  # @param {type:"number"}
SEEDS = [0, 1, 2, 3, 4]  # @param {type:"raw"}
SAVE_TO_DRIVE = False  # @param {type:"boolean"}

from io import BytesIO
from pathlib import Path
from runpy import run_path
from urllib.request import urlopen
from zipfile import ZipFile

from google.colab import drive, files

REVISION = "3a7f954c42ab12b3783da40057eef222df7bc7c8"
workspace = Path("/content/pdpcr_colab")
source = workspace / f"colab_test-{REVISION}"
runner = source / "run_tsr_colab.py"
if not runner.is_file():
    print("Đang tải mã benchmark từ GitHub...", flush=True)
    url = f"https://codeload.github.com/dhducpro/colab_test/zip/{REVISION}"
    with urlopen(url, timeout=60) as response, ZipFile(BytesIO(response.read())) as archive:
        archive.extractall(workspace)

output_root = workspace / "results"
if SAVE_TO_DRIVE:
    drive.mount("/content/drive")
    output_root = Path("/content/drive/MyDrive/PDPCR_TSR_benchmark")

benchmark = run_path(str(runner))
result_zip = benchmark["run_colab"](
    TIME_LIMIT, SEEDS, "thao_elite", output_root, REVISION,
    instances=benchmark["N50_INSTANCES"],
)
files.download(str(result_zip))
